In [39]:
from typing import Union, Optional, Callable
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Lab 05: Hidden Markov Models

In this session, we will implement the algorithms presented in the lecture as solutions to the various tasks related to the Hidden Markov Models (HMMs): Filtering, Smoothing and Decoding.

<hr>

## Part 0: NumPy Crash Course
Before stating the tasks, we have a quick crash course on the basics of NumPy, as it will be needed for the tasks in this lab, as well as future ones.

### Introduction

Vectors, matrices, etc. are all considered **arrays** in NumPy. A common way of creating arrays is by calling `np.array` with a (nested) list, like so:
```python
np.array([0.1, 0.3])
```
This creates a 1-dimensional array, or vector. A 2-dimensional array, or matrix, can be created analogously:
```python
np.array([[0.1, 0.3], [0.5, 0.7]])
```
In general, you can create arrays of arbitrarily many dimensions. These are commonly called **tensors**. To learn more about the dimensions of a particular array/tensor, you can look at its **shape**:

In [40]:
a = np.array([[0.1, 0.3], [0.5, 0.7]])
a.shape

(2, 2)

The output indicates that the array has two dimensions, which corresponds to the length of the tuple. The first dimension has length 2, and the second dimension also has length 2. Hence, this is a $2 \times 2$ matrix.

### Indexing
To index a particular value of a tensor, write the index for each dimension inside of the brackets:

In [41]:
a[1,1]

np.float64(0.7)

This returns the element in the 2nd row and 2nd column, because Python starts indexing at 0. If you want the entire second row, you can write:

In [42]:
a[1,:]

array([0.5, 0.7])

Similarly, you can write

In [43]:
a[:,1]

array([0.3, 0.7])

to get the second column. Colons are implicitly appended at the end if the number of indices does not match the dimensionality of the tensor. Hence, we could have also written `a[1]` to get the second row. But this wouldn't work for the second column.

### Operations
If you have two tensors, `a` and `b` of the same shape, the common arithmetic operations (`+`, `*`, `/`) are performed element-wise. For example:

In [44]:
a = np.array([1., 2.])
b = np.array([-0.5, 0.5])

In [45]:
a + b

array([0.5, 2.5])

In [46]:
a * b

array([-0.5,  1. ])

In [47]:
a / b

array([-2.,  4.])

If the shapes do not match, NumPy will try to perform **broadcasting**. The idea is that these operations should still be possible, when the intended behavior makes sense. For instance, multiplying a vector by a scalar (i.e., a tensor of shape `(1,)`) should be possible: every element of the vector should be multiplied by a scalar.

Broadcasting works by scanning through the dimensions of both tensors from _back to front_. Dimensions _match_ if they have the same length, or if one of them has length $1$. If NumPy finds non-matching dimensions before the least-dimensional array runs out of dimensions, a `ValueError` is thrown -- the operation is not possible. The resulting array will have as many dimensions has the highest-dimensional array. Here is a non-trivial example:

In [48]:
a = np.array([[1., 2.],[3., 4.]])  # shape (2, 2)
b = np.array([[[1., 2.]], [[3., 4.]], [[5., 6.]]])  # shape (3, 1, 2)

# The resulting array will have shape (3, 2, 2)
a * b

array([[[ 1.,  4.],
        [ 3.,  8.]],

       [[ 3.,  8.],
        [ 9., 16.]],

       [[ 5., 12.],
        [15., 24.]]])

There is one more important operation, namely **matrix multiplication**. In NumPy, it is denoted by `@`. It can also be used for matrix-vector or vector-matrix multiplication.

### Transposition and Reshaping
Sometimes, we want to use the values of an array, but with a different order of dimensions. A well-known example is (matrix) transposition:
$$\begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix} \quad\rightsquigarrow\quad \begin{pmatrix} 1 & 3 \\ 2 & 4 \end{pmatrix}.$$
This effectly corresponds to reversing the order of the dimensions. This is what `a.transpose()` does. A shorthand is `a.T`.

A different form of manipulating dimensions is given by **reshaping**. Using the method `a.reshape(...)`, you can provide the lengths of your desired new dimensions, as long as their product is equal to the product of the original dimensions. Optionally, you can replace one dimension by `-1`, indicating that the length of this dimension should be inferred by NumPy. For example:

In [49]:
a = np.array([[[1., 2.]], [[3., 4.]], [[5., 6.]]])  # shape (3, 1, 2)
a.reshape(-1)  # equivalent to a.flatten()

array([1., 2., 3., 4., 5., 6.])

In [50]:
a.reshape(2, 3)

array([[1., 2., 3.],
       [4., 5., 6.]])

In [51]:
a.reshape(6, 1)

array([[1.],
       [2.],
       [3.],
       [4.],
       [5.],
       [6.]])

In [52]:
a.reshape(3, -1)

array([[1., 2.],
       [3., 4.],
       [5., 6.]])

### Documentation
Finally, NumPy has a very good documentation, which you can consult while solving the tasks: https://numpy.org/doc/stable/reference/index.html
<hr>

## Part 1: Defining the HMM

The structure of a Hidden Markov Model (HMM) can be expressed as follows in Python:

In [53]:
from hmm import HiddenMarkovModel, make_grid_hmm

In this session, we will consider two simple HMMs.

The first HMM describes the behaviour of a person (named Bob) depending on the weather. Bob can choose his activity depending on the weather (sunny or rainy): either going for a walk, shopping or cleaning inside. 

The weather transitions are as follows: 
* When it is raining on some day, it is raining the following day with probability 0.7
* When it is sunny on some day, it is sunny the following day with probability 0.6

Bob decides his daily activity as follows:
* When it is raining on some day, Bob goes for a walk with probability 0.1, goes shopping with probability 0.4 and stays inside cleaning with probability 0.5. 
* When it is sunny on some day, Bob goes for a walk with probability 0.6, goes shopping with probability 0.3 and stays inside cleaning with probability 0.1. 


### Question 1

Implement the corresponding HMM. 

In [54]:
weather_hmm = HiddenMarkovModel(S=np.array(["rainy", "sunny"]), O=np.array(["walk", "shop", "clean"]), A=np.array([[0.7,0.3], [0.4, 0.6]]), psi=np.array([[0.1,0.4,0.5],[0.6,0.3,0.1]]), psi_fun = None, pi = np.array([0.5, 0.5]))

The second HMM corresponds to some artificial situation where we observe the (noisy) sum of two integers. These two integers are chosen through a random walk on a bounded subset of $\mathbb Z^2$.

The HMM is specified as follows:
* The state space $\mathcal{S}$ is a set of tuples corresponding to a finite grid: $\mathcal{S} = \lbrace - N, \dotsc, N \rbrace^2$ for some integer $N > 0$
* The state transition corresponds to a random walk on the grid: from a state $(i,j)$, one can transition with equiprobability to any state $(i^\prime, j^\prime)$, if $\lvert i' - i\rvert + \lvert j' - j\rvert = 1$
* The initial distribution is uniform over the grid
* The observation space $\mathcal{O}$ is the space of real numbers: $\mathcal{O} = \mathbb{R}$
* The emission probabilities are normal distributions: $p(x \mid z = (i,j)) = \mathcal{N}(x; i + j, \sigma^2)$

This HMM is already implemented for you.

In [55]:
N = 2
sigma = 0.1
grid_hmm = make_grid_hmm(N, sigma)

## Part 2: Filtering

The goal of filtering is to compute the probabilities $\alpha_t(j) = p(z_t = j \mid x_1, \dotsc, x_t)$. This is done throughout the forwards algorithm.

### Question 2 (*)

a) Implement the forwards algorithm.<br>

In [62]:
def forward(hmm: HiddenMarkovModel, x: np.ndarray):
    T = len(x)
    N = len(hmm.pi)
    psi = hmm.get_psi(x)
    alpha = np.zeros((T, N))
    alpha[0, :] = psi[:, 0] * hmm.pi
    for t in range(1, T):
        alpha[t, :] = psi[:, t] * (hmm.A.T @ alpha[t-1, :]) 
    prob = np.sum(alpha[T-1, :])
    return alpha, prob

b) We consider the following observations for the two tasks. Apply the forward algorithm to these two observations. Do the outcomes look consistent? Compute also the probability $p(x_1, \dotsc, x_T)$.

In [63]:
# 1. Observations for weather HMM
x_weather = [0, 0, 2, 1, 0]

forward(weather_hmm, x_weather)

# 2. Observations for grid HMM
# spiral = [(0,0), (0,1), (-1,1), (-1,0), (-1,-1), (0,-1), (1,-1), (1,0), (1,1), (1,2), (0,2), (-1,2), (-2,2), (-2,1), (-2,0), (-2,-1), (-2,-2), (-1,-2), (0,-2), (1,-2), (2,-2), (2,-1), (2,0), (2,1), (2,2)]
# x_grid = [norm.rvs(loc = i+j, scale = sigma) for (i,j) in spiral]

# print("Grid: ", forward(grid_hmm, x_grid))

(array([[0.05      , 0.3       ],
        [0.0155    , 0.117     ],
        [0.028825  , 0.007485  ],
        [0.0092686 , 0.00394155],
        [0.00080646, 0.00308731]]),
 np.float64(0.0038937699999999995))

The observations for the grid HMM are generated from a spiral which starts at the origin $(0,0)$ and moves outward, visiting every state. From this, the sums are computed and random noise is applied.

## Part 3: Smoothing

The goal of smoothing is to compute the probabilities $\gamma_t(j) = p(z_t = j \mid x_1, \dotsc, x_T)$. Smoothing is solved using the forwards-backwards algorithm.

### Question 3 (*)

a) Implement the forwards-backwards algorithm.<br>
b) Test the algorithm on the $x$'s we introduced in the previous part.

In [ ]:
def backward(hmm: HiddenMarkovModel, x: np.ndarray):
    T = len(x)
    N = len(hmm.pi)
    psi = hmm.get_psi(x)
    beta = np.zeros((T, N))
    beta[T-1, :] = np.ones(1)
    for t in range(T-1, 1, -1):
        beta[t-1, :] = hmm.A @ (psi[:, t] * beta[t, :])

def forward_backward(hmm: HiddenMarkovModel, x: np.ndarray):
    pass  # TODO

## Part 4: Decoding

Decoding corresponds to the task of finding the optimal sequence of states explaining an observation. 

A first method for decoding consists in maximizing the posterior marginals. This corresponds to selecting, for each time step $t$, the state $j$ maximizing $p(z_t = j \mid x_1, \dotsc, x_T)$. 

### Question 4 (*)

Implement the posterior marginals maximization (MPP). Compute the optimal sequence of states for the given two sequences of observations. 

In [ ]:
def mpp_decoding(hmm: HiddenMarkovModel, x: np.ndarray):
    gammas = forward_backward(hmm, x)
    z_mpp = ...  # TODO: Your code here
    return z_mpp

The problem with the MPP decoding is that it does not maximize the probability of the whole sequence, and might therefore lead to sub-optimal choices. For this reason, why prefer using the Viterbi algorithm, which relies on dynamic programming to compute the maximum a posteriori estimator of the latent states $z_1, \dotsc, z_T$.

### Question 5 (*)

Implement the Viterbi algorithm. Compare the results with the MPP decoding. 

In [ ]:
def viterbi(hmm: HiddenMarkovModel, x: np.ndarray):
    T = len(x)
    psi = hmm.get_psi(x)
    
    # TODO

### Question 6

Look at the solution returned by the Viterbi algorithm for the grid HMM. How well does it recover the trajectory that generated the observations? Can you explain the behavior?